In [2]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter, Language
from langchain.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.memory import VectorStoreRetrieverMemory
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from datetime import datetime, timezone
from langchain.memory import ConversationBufferMemory, VectorStoreRetrieverMemory, CombinedMemory
from langchain.schema.runnable import RunnableParallel, RunnableLambda, RunnablePassthrough

In [4]:
load_dotenv()

True

In [5]:
url = 'https://www.flipkart.com/motorola-motobook-60-full-metal-oled-i5-14th-gen-intel-core-5-series-2-210h-16-gb-512-gb-ssd-windows-11-home-14irh10r-thin-light-laptop/p/itm9a50f9400e0e0?pid=COMHAUZWVNJSFAMN&otracker=wishlist&lid=LSTCOMHAUZWVNJSFAMNCQMECQ&fm=organic&iid=eb8ce955-00a1-4fdc-bd17-e3cc38eb95b2.COMHAUZWVNJSFAMN.PRODUCTSUMMARY&ppt=hp&ppn=homepage&ssid=txin2y7f340000001758805452784'

In [ ]:
llm = HuggingFaceEndpoint(
    model="meta-llama/Llama-3.3-70B-Instruct",
    task="text-generation",
    temperature=0.3
)
model = ChatHuggingFace(llm=llm)

d:\anaconda3\envs\genai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Loading Raw HTML from the Webpage

loader = WebBaseLoader(url)
raw_html = loader.scrape()                # returns BeautifulSoup object

In [8]:
# Remove unnecessary tags from raw HTML

for tag in raw_html([
    "script", "style", "nav", "footer", "header", "aside", "meta", "link",
    "noscript",   # fallback content, usually redundant
    "iframe",     # embedded ads, videos
    "form",       # login/signup/contact forms
    "input", "button", "select", "textarea",  # form fields
    "svg", "canvas",  # icons, graphics
    "img",       # images (unless you want alt text)
    "video", "audio", "source", "track",  # media elements
    "advertisement", "ads",  # ad containers (if present as tags)
]):
    tag.decompose()

clean_html = str(raw_html)

In [9]:
# Remove all attributes from the tags

soup = BeautifulSoup(clean_html, "html.parser")
for tag in soup.find_all(True):                    # True = all tags
    tag.attrs = {}

clean_html = str(soup)

In [10]:
# Split each section by HTML-aware splitter

html_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.HTML,
    chunk_size=5000,
    chunk_overlap=1000
)

html_chunks = html_splitter.split_text(clean_html)

print(f"Number of HTML-aware chunks: {len(html_chunks)}")

Number of HTML-aware chunks: 9


In [11]:
html_chunks[4][:500]

'<div><div>Connectivity Features</div><table><tbody><tr><td>Bluetooth</td><td><ul><li>Yes</li></ul></td></tr></tbody></table></div></div></div></div></div><div><div><div><span>Buy Together and Save upto 10%</span></div><div><div><div><div><div><a><div><div><div></div></div></div></a><div><div><a>MOTOROLA Motobook 60 Full Metal OLED (i5 14th Gen) Intel Core 5 (...</a></div><div><span><div>4.4</div></span><span>(1,111)</span></div><div><div><div>₹52,999</div><div>₹<!-- -->93,690</div><div><span>43%'

In [12]:
# Replace all HTML tags with a common separator for better splits

clean_chunks = []
for chunk in html_chunks:
    soup = BeautifulSoup(chunk, "html.parser")
    clean_chunk = soup.get_text(separator="---", strip=True)
    if len(clean_chunk) > 5:
        clean_chunks.append(clean_chunk)


In [13]:
print(sorted([len(chunk) for chunk in clean_chunks]))     # length of chunks
print(clean_chunks[-2][:500])

[500, 1856, 1882, 2061, 2067, 2455, 2476, 2820]
All 125 reviews---Questions and Answers---Q:---How is the Battery life......---A:---For normal if you are using YouTube or powerpoint or surfing it is around 6 hrs approx. Heavy task like editing and all 2 hrs to 3hrs.---Anonymous---Certified Buyer---33---8---Report Abuse---Q:---Does it have lifetime ms office---A:---Yes---Demon King---Certified Buyer---2---0---Report Abuse---Read other answers---Q:---I saw many negative reviews on sound quality, how it is?---A:---Speakers aren't that loud but t


In [14]:
# Final splitter for basic text

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["---", "\n\n\n", "\n\n", "\n", ".", " ", ""]
)
modified_chunks = []
for text in clean_chunks:
    new_chunks = splitter.split_text(text)
    modified_chunks.extend(new_chunks)


# chunks = []
chunks = [text.replace("---", " ") for text in modified_chunks]
# for text in modified_chunks:
#     new_text = text.replace("---", " ")
#     chunks.append(new_text)

print("Length of the final chunks: ", len(chunks))
print(sorted([len(chunk) for chunk in chunks]))
print(chunks[-2])

Length of the final chunks:  23
[171, 251, 287, 408, 450, 523, 574, 657, 789, 826, 863, 865, 868, 870, 879, 881, 903, 924, 927, 929, 934, 936, 941]
 Certified Buyer 1 1 Report Abuse Q: does it have C port for charging? A: Yes Rahul  Khatkar Certified Buyer 1 1 Report Abuse Q: Ms office lifetime are not A: With a Microsoft account we can use MS Office basics...

If you need the full fledged options, either to use cracked or purchase Office 365... vignesh baskar Certified Buyer 1 2 Report Abuse All questions + Safe and Secure Payments. Easy returns. 100% Authentic products. You might be interested in Physical Min. 50% Off Shop Now External SSD Min. 50% Off Shop Now


In [15]:
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_texts(chunks, embeddings)

In [16]:
# MMR Retriever
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 5, 'fetch_k': 10, 'lambda_mult': 0.7}
)

# Multiquery Retriever
retriever = MultiQueryRetriever.from_llm(
    retriever=mmr_retriever,
    llm=model
)


In [17]:
# memory_store = FAISS.from_texts([""], embeddings)
# memory = VectorStoreRetrieverMemory(retriever=mmr_retriever)

In [18]:
# Memory setup
buffer_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

memory_store = FAISS.from_texts([""], embeddings)
memory_retriever = memory_store.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 5, 'fetch_k': 10, 'lambda_mult': 0.7}
)
vector_memory = VectorStoreRetrieverMemory(retriever=memory_retriever)

memory = CombinedMemory(memories=[buffer_memory, vector_memory])

C:\Users\pc\AppData\Local\Temp\ipykernel_12024\365941363.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  buffer_memory = ConversationBufferMemory(
C:\Users\pc\AppData\Local\Temp\ipykernel_12024\365941363.py:12: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  vector_memory = VectorStoreRetrieverMemory(retriever=memory_retriever)
C:\Users\pc\AppData\Local\Temp\ipykernel_12024\365941363.py:14: UserWarning: When using CombinedMemory, input keys should be so the input is known.  Was not set on chat_memory=InMemoryChatMessageHistory(messages=[]) return_messages=True memory_key='chat_history'
  memory = CombinedMemory(memories=[buffer_memory, vector_memory])


In [19]:
ans = retriever.invoke('what are the reviews ?')
len(ans)

13

In [20]:
prompt_template = """
You are a helpful assistant.
You give concise answers.

Conversation history (retrieved from memory):
{history}

Relevant Webpage Content context:
{context}

User Query: {query}

Answer clearly using both conversation history and webpage content context.
If the user asks for a summary/overview, summarize the whole webpage content.
Don't mention where or how in the hostory you are getting the information from, just answer what the user needs.
If the query conntains pronouns that you cannot interpret what is referring to, then consider the most recent conversation history to interpret them.
If the context doesn't contain the relevant information to answer the user query, then say something like Webpage doesn't contain the relevant information.
"""

In [21]:
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["history", "context", "query"]
)

In [22]:
def format_docs(retrieved_docs):
    return "\n\n".join(
        f"[Chunk {i}] {doc.page_content}" for i, doc in enumerate(retrieved_docs, 1)
    )

In [23]:
parser = StrOutputParser()

In [24]:
# def get_history(query: str):
#     history = memory.load_memory_variables({'input': query}).get("history", "")
#     return history

def get_history(query: str):
    history_vars = memory.load_memory_variables({'input': query})
    history = history_vars.get("chat_history", [])
    if isinstance(history, list):
        formatted_history = "\n".join(
            f"{msg.type.upper()}: {msg.content}" for msg in history
        )
    else:
        formatted_history = history
    return formatted_history


In [25]:
parallel_chain = RunnableParallel({
    'history': RunnableLambda(get_history),
    'context': retriever | RunnableLambda(format_docs),
    'query': RunnablePassthrough()
})

In [26]:
main_chain = parallel_chain | prompt | model | parser

In [27]:
def ask(query: str):
    answer = main_chain.invoke(query)

    # Save to buffer memory (only output)
    buffer_memory.save_context({"input": query}, {"output": answer})

    # Save to vector memory with timestamp
    vector_memory.save_context(
        {"input": query},
        {"output": answer, "timestamp": datetime.now(timezone.utc).isoformat()}
    )

    return answer

In [ ]:
def reset_history():
    # Clear buffer memory
    buffer_memory.clear()


    memory_store = FAISS.from_texts([""], embeddings)
    memory_retriever = memory_store.as_retriever(
        search_type="mmr",
        search_kwargs={'k': 5, 'fetch_k': 10, 'lambda_mult': 0.7}
    )
    vector_memory = VectorStoreRetrieverMemory(retriever=memory_retriever)

    memory.memories = [buffer_memory, vector_memory]

In [29]:
# reset_history()

In [30]:
ask("what is the price of this product ?")

'The price of this product is ₹52,999, which is 43% off from the original price of ₹93,690.'

In [31]:
ask("what does the reviews say about price ?")

'The reviews suggest that the price of the laptop is reasonable, with one buyer mentioning they got it for ₹47,000 after a discount, and another saying it\'s "worth every penny" at the price of ₹52,999, which is already 43% off from the original price of ₹93,690.'

In [32]:
ask("what is the most negative review about this product ?")

'The most negative review about this product states that it\'s "only good for web browsing and watching videos" and that the sound quality is "not good". The reviewer mentions that their phone has louder sound than this laptop, and it\'s not smooth for Excel work, making it not suitable for office workers who do a lot of Excel work. The reviewer suggests that students or those preparing for job exams might find it suitable, but recommends considering a higher variant for better performance.'

In [33]:
ask("is there anything positive this reviewer said ?")

'The reviewer mentioned that the product is "good for web browsing and watching videos" and suggested it for students or those preparing for job exams, which implies some positive aspects of the laptop.'

In [34]:
ask("what questions have i asked to you until now?")

'You have asked the following questions until now:\n\n1. What is the price of this product?\n2. What does the reviews say about price?\n3. What is the most negative review about this product?\n4. Is there anything positive this reviewer said?\n5. What questions have I asked to you until now?'